In [0]:
%sql
CREATE CATALOG IF NOT EXISTS vinkoscon
MANAGED LOCATION 'abfss://processeddatabricks@strfilese.dfs.core.windows.net/vinkos_managed/';

In [0]:
%sql
-- Forzamos el uso de tu nuevo catálogo
--USE CATALOG vinkoscon;

-- Creamos el esquema
--CREATE SCHEMA IF NOT EXISTS bronze;

-- Creamos la tabla (CTAS)
--CREATE TABLE vinkoscon.bronze.visitas_raw
--PARTITIONED BY (create_date)
--AS SELECT *  FROM azure_sql_vinkos_catalog.dbo.usuarios_visitas_raw where create_date = date_format(current_timestamp());

INSERT INTO vinkoscon.bronze.visitas_raw (
email
,fechaPrimeraVisita
,fechaUltimaVisita
,visitasTotales
,visitasAnioActual
,visitasMesActual
,jyv
,Badmail
,Baja
,Fecha_envio
,Fecha_open
,Opens
,Opens_virales
,Fecha_click
,Clicks
,Clicks_virales
,Links
,IPs
,Navegadores
,Plataformas
,errores
,create_date
)
SELECT 
email
,fechaPrimeraVisita
,fechaUltimaVisita
,visitasTotales
,visitasAnioActual
,visitasMesActual
,jyv
,Badmail
,Baja
,Fecha_envio
,Fecha_open
,Opens
,Opens_virales
,Fecha_click
,Clicks
,Clicks_virales
,Links
,IPs
,Navegadores
,Plataformas
,errores
,cast(create_date as string) as create_date
FROM azure_sql_vinkos_catalog.dbo.usuarios_visitas_raw where cast(create_date as date) = current_date();

In [0]:
from pyspark.sql.functions import col, regexp_like, lit, trim ,to_timestamp ,to_date, date_format

#limpiamos el correo, aquel que no cumpla lo desechamos
df = spark.read.table("vinkoscon.bronze.visitas_raw")
emails_valido = df.filter(col("email").rlike(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')).select("*")
sin_duplicados= emails_valido.dropDuplicates(['email'])
 
df_final = sin_duplicados.withColumn(
"fechaPrimeraVisita", 
    to_timestamp("fechaPrimeraVisita", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "fechaUltimaVisita", 
    col("fechaUltimaVisita").cast("string"))


 

In [0]:
%sql
-- Creamos el esquema
CREATE SCHEMA IF NOT EXISTS silver;